## Data Preparation

You should prepare the following before running this step. Please refer to the `example_data` folder for guidance:

1. **NIfTI image of fixed CT** resampled to [1,1,1]mm^3 voxel size
   - please refer to ```example_data/fixed_CT/00014689/img_1mm.nii.gz``` for reference

---

## Motion simulation

We use this script to simulate motion-corrupted image from motion-free fixed CT scans. We mimic the portable CT (with narrower z-coverage that requires multiple gantry rotations to cover the head). please read our paper for more info.

The simulation output is resampled to [1,1,2.5]mm^3 for next steps.

### Docker environment
Please use `docker/docker_tensorflow`, it will build a tensorflow-based container

make sure you have `https://github.com/zhennongchen/CTProjector.git` installed


In [1]:
import os
import sys
sys.path.append('/workspace/Documents')  
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
import glob as gb
import nibabel as nb 
import math
import pandas as pd
import os
from skimage.measure import block_reduce
import Diffusion_for_CT_motion.motion_simulation.ct_basic as ct
import Diffusion_for_CT_motion.functions_collection as ff
import Diffusion_for_CT_motion.motion_simulation.transformation as transform
import Diffusion_for_CT_motion.utils.Data_processing as dp
import ct_projector.projector.cupy as ct_projector

main_path = '/mnt/camca_NAS/diffusion_ct_motion'

/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### set some default parameters

In [2]:
amplitude_max_severe = 10 
displacement_max_severe = 6
amplitude_max_mild = 5 
displacement_max_mild = 3

motion_freq = 0.5 
severe_freq = 0.5
double_skull_freq = 0.25 

change_direction_limit = 2
CP_num = 5

geometry = 'fan'
total_view = 1400  
gantry_rotation_time = 500 
view_increment = 28

### define patient list

In [3]:
# define the patient list
patient_list = ff.find_all_target_files(['*/*'],os.path.join(main_path,'example_data/fixed_CT'))

data_folder = os.path.join(main_path, 'example_data/fixed_CT')
save_folder = os.path.join(main_path, 'example_data/simulations')
ff.make_folder([save_folder])

### do for a static simulation 
meaning there's no simulation at all, we just re-forward-and-backward project our data using this simulated system to remove the discrepancy caused by system

In [11]:
for i in range(len(patient_list)):

    patient_id =os.path.basename(os.path.dirname(patient_list[i]))
    patient_subid = os.path.basename(patient_list[i])
    print('\n',i, patient_id, patient_subid)

    save_folder_patient = os.path.join(save_folder, patient_id, patient_subid)
    ff.make_folder([os.path.join(save_folder, patient_id), save_folder_patient])

    img_file = ff.find_all_target_files(['img_1mm.nii.gz'],os.path.join(data_folder, patient_id, patient_subid))
    if len(img_file) != 1:
        ValueError('no raw data')
    
    img,spacing,img_affine = ct.basic_image_processing(img_file[0])
    print('nib image shape: ',img.shape, ' spacing: ',spacing)

    # define projectors
    img = img[np.newaxis, ...]
    projector = ct.define_forward_projector(img,spacing,total_view)
    fbp_projector = ct.backprojector(img,spacing)
    
    # very important - make sure that the arrays are saved in C order
    cp.cuda.Device(0).use()
    ct_projector.set_device(0)

    t = np.linspace(0, gantry_rotation_time, CP_num, endpoint=True)
    # create folder
    random_folder = os.path.join(save_folder_patient,'static')
    ff.make_folder([random_folder, os.path.join(random_folder,'image_data')])

    # gantry coverage = 1cm, which means it rotates once for every 1cm interval in the z-axis
    # set a sga reference
    sga_list = []; sga_reference = int(np.random.uniform(0,90)) # starting gantry rotation
    # second, find out how many rotations we need
    print('spacing: ', spacing, ' img shape: ', img.shape)
  
    rotation_num = 1
    slice_coverage = int(spacing[0] * img.shape[1] // 10) * 10
    print('rotation_num: ', rotation_num, ' slice_coverage: ', slice_coverage)
    motion_status = [False] 
    print('motion_status: ', motion_status)
        
    amplitude_collect = np.zeros([rotation_num, CP_num, 6])
    recon = np.zeros([slice_coverage * rotation_num,img.shape[2],img.shape[3]])
    projection = np.zeros([slice_coverage * rotation_num, total_view, 1 , projector.nu])

    for rot_n in range(0,rotation_num):
        # first set the SGA:
        sga = sga_reference + int(np.random.uniform(-5,5))
        sga = max(0, sga); sga = min(90, sga)
        sga_list.append(sga)
        # set the partial image:
        slice_start = rot_n * slice_coverage
        slice_end = (rot_n + 1) * slice_coverage
        slice_prior = max(0, slice_start - slice_coverage * 2)
        slice_post = min(img.shape[1], slice_end + slice_coverage * 2)
        print('slice start: ', slice_start, ' slice end: ', slice_end, ' slice prior: ', slice_prior, ' slice post: ', slice_post)
        img_partial = img[:,slice_prior:slice_post,:]

        # set the motion 
        # first: determine whether in this rotation we have motion or static, and whether it's severe or mild
     
        severe_motion = False
        amplitude_max_tem_t, displacement_max_tem_t, amplitude_max_tem_r, displacement_max_tem_r = 0, 0, 0, 0
            
        offset_values = [0] * 6
            
        # generate the motion
        # translation
        amplitude_txty_mm = transform.motion_control_point_generation(2, CP_num, amplitude_max = amplitude_max_tem_t, displacement_max = displacement_max_tem_t, change_direction_limit = change_direction_limit, offset_value = offset_values[0:2], print_result =False)
        amplitude_tx_mm = amplitude_txty_mm[:,0]
        amplitude_ty_mm = amplitude_txty_mm[:,1]
        amplitude_tz_mm = transform.motion_control_point_generation(1, CP_num, amplitude_max = 0, displacement_max = 0, change_direction_limit = change_direction_limit, offset_value = offset_values[2], print_result =False)[:,0]
        # rotations
        while True:
            amplitude_rxryrz_degree = transform.motion_control_point_generation(3, CP_num, amplitude_max = amplitude_max_tem_r, displacement_max = displacement_max_tem_r, change_direction_limit = change_direction_limit, offset_value = offset_values[3:6], print_result =False)
            amplitude_rx_degree = amplitude_rxryrz_degree[:,0]
            amplitude_ry_degree = amplitude_rxryrz_degree[:,1]
            amplitude_rz_degree = amplitude_rxryrz_degree[:,2]
            if np.max(abs(amplitude_rx_degree - amplitude_rx_degree[0]))+ np.max(abs(amplitude_ry_degree - amplitude_ry_degree[0])) <= 6:
                break
       

        print('amplitude_tx_mm: ', amplitude_tx_mm)
        print('amplitude_ty_mm: ', amplitude_ty_mm)
        print('amplitude_tz_mm: ', amplitude_tz_mm)
        print('amplitude_rx_degree: ', amplitude_rx_degree)
        print('amplitude_ry_degree: ', amplitude_ry_degree)
        print('amplitude_rz_degree: ', amplitude_rz_degree)
            
        # save motion paramaters
        collect = np.stack([amplitude_tx_mm, amplitude_ty_mm, amplitude_tz_mm, amplitude_rx_degree, amplitude_ry_degree, amplitude_rz_degree], axis = 1)
        amplitude_collect[rot_n,...] = collect
            
        # prepare spline fit
        spline_tx = transform.interp_func(t, np.asarray([i/spacing[1] for i in amplitude_tx_mm]))
        spline_ty = transform.interp_func(t, np.asarray([i/spacing[2] for i in amplitude_ty_mm]))
        spline_tz = transform.interp_func(t, np.asarray([i/spacing[0] for i in amplitude_tz_mm]))
        spline_rx = transform.interp_func(t,np.asarray([i / 180 * np.pi for i in amplitude_rx_degree]))
        spline_ry = transform.interp_func(t,np.asarray([i / 180 * np.pi for i in amplitude_ry_degree]))
        spline_rz = transform.interp_func(t,np.asarray([i / 180 * np.pi for i in amplitude_rz_degree]))

        angles = ff.get_angles_zc(total_view, 360 ,sga)


        # generate forward projection
        projection_partial = ct.fp_w_spline_motion_model(img_partial, projector, angles, spline_tx, spline_ty, spline_tz, spline_rx, spline_ry, spline_rz, geometry, total_view = total_view, gantry_rotation_time = gantry_rotation_time, slice_num = None, increment = view_increment, order = 3)
        projection_partial = projection_partial[slice_start - slice_prior : slice_end - slice_prior,...]
            
        # generate backprojection
        recon_partial = ct.filtered_backporjection(projection_partial,angles,projector,fbp_projector, geometry, back_to_original_value=True)
        recon[slice_coverage * (rot_n) : slice_coverage * (rot_n + 1),...] = recon_partial


    parameter_file = os.path.join(random_folder,'motion_parameters.npy')
    np.save(parameter_file, np.array([[amplitude_collect],[sga_list], [t], [total_view], [gantry_rotation_time]], dtype=object))
           
    # save recon
    recon_nb_image = np.rollaxis(recon,0,3)  
    nb.save(nb.Nifti1Image(recon_nb_image,img_affine), os.path.join(random_folder,'image_data','recon.nii.gz'))

    ####### resample recon to [1,1,2.5]mm
    img_1mm_file = nb.load(os.path.join(random_folder,'image_data','recon.nii.gz'))
    img_1mm = img_1mm_file.get_fdata()
    # original pixel dim
    pixel_dim = img_1mm_file.header.get_zooms()[:3]
    print('original pixel dim:', pixel_dim)
     
    # first, resample to z=1.25mm using interpolation, so that 2.5mm can be achieved by averaging
    new_dim = [1,1,1.25]
    hr_resample = dp.resample_nifti(img_1mm_file, order=3,  mode = 'nearest',  cval = np.min(img_1mm_file.get_fdata()), in_plane_resolution_mm=new_dim[0], slice_thickness_mm=new_dim[-1])
    hr_resample = nb.Nifti1Image(hr_resample.get_fdata(), affine=hr_resample.affine, header=hr_resample.header)

    # print('last shape:', hr_resample.get_fdata().shape)
    print('new pixel dim:', hr_resample.header.get_zooms()[:3])

    # now, do for z resampling (using averaging)
    img_data = hr_resample.get_fdata()
    affine = hr_resample.affine
    pixel_dim = hr_resample.header.get_zooms()[:3]
    header = hr_resample.header

    new_z_res = 2.5

    slice_factor = new_z_res // pixel_dim[-1]

    # Calculate the new shape
    new_shape = list(img_data.shape)
    new_shape[2] = int(img_data.shape[2] // slice_factor)
    print('new shape:', new_shape)  
    # Initialize the resampled data
    resampled_data = np.zeros(new_shape)

    for i in range(new_shape[2]):
        start_slice = int(i * slice_factor)
        end_slice = int((i + 1) * slice_factor)

        resampled_data[:, :, i] = np.mean(img_data[:, :, start_slice:end_slice], axis=2)

    # Update the affine and header for the new voxel size
    new_affine = affine.copy()
    new_affine[2, 2] = affine[2, 2] * slice_factor
    new_header = header.copy()
    new_header.set_zooms([1,1, pixel_dim[-1] * slice_factor])
    
    # Create and save the new NIfTI image
    resampled_img = nb.Nifti1Image(resampled_data, new_affine, new_header)
    # print new pixel dim:
    print('final pixel dim:', resampled_img.header.get_zooms()[:3])

    nb.save(resampled_img, os.path.join(random_folder,'image_data','recon_resample_avg.nii.gz'))



 0 00014689 0000455416
nib image shape:  (161, 239, 239)  spacing:  [1. 1. 1.]
original pixel dim: (1.0, 1.0, 1.0)
new pixel dim: (1.0, 1.0, 1.25)
new shape: [239, 239, 64]
final pixel dim: (1.0, 1.0, 2.5)


### do the motion simulation
the key thing is that it requires several gantry rotations to cover the head, and the motion may or may not occur in each rotation

In [12]:
L = np.arange(1,2)

for i in range(len(patient_list)):

    patient_id =os.path.basename(os.path.dirname(patient_list[i]))
    patient_subid = os.path.basename(patient_list[i])
    print('\n',i, patient_id, patient_subid)


    save_folder_patient = os.path.join(save_folder, patient_id, patient_subid)
    ff.make_folder([os.path.join(save_folder, patient_id), save_folder_patient])

    img_file = ff.find_all_target_files(['img_1mm.nii.gz'],os.path.join(data_folder, patient_id, patient_subid))
    if len(img_file) != 1:
        ValueError('no raw data')
    
    img,spacing,img_affine = ct.basic_image_processing(img_file[0])
    print('nib image shape: ',img.shape, ' spacing: ',spacing)

    # define projectors
    img = img[np.newaxis, ...]
    projector = ct.define_forward_projector(img,spacing,total_view)
    fbp_projector = ct.backprojector(img,spacing)
    
    # very important - make sure that the arrays are saved in C order
    cp.cuda.Device(0).use()
    ct_projector.set_device(0)

    # load the simulated static image as reference
    if os.path.isfile(os.path.join(main_path, 'simulations',patient_id, patient_subid, 'static','image_data','recon.nii.gz')) == 0:
        static_ref = None
    else:
        static_ref= nb.load(os.path.join(main_path, 'simulations',patient_id, patient_subid, 'static','image_data','recon.nii.gz')).get_fdata()
        static_ref = np.rollaxis(static_ref,2,0)
        print('static ref shape: ', static_ref.shape)
    print('static_ref is None? ', static_ref is None)

    for random_i in L:
        t = np.linspace(0, gantry_rotation_time, CP_num, endpoint=True)
        # create folder
        random_folder = os.path.join(save_folder_patient,'random_' +str(random_i))
        ff.make_folder([random_folder, os.path.join(random_folder,'image_data')])
        print('\n',random_i  , 'random')


        # # gantry coverage = 1cm, which means it rotates once for every 1cm interval in the z-axis
        # # set a sga reference
        # sga_list = []; sga_reference = int(np.random.uniform(0,90))
        # # second, find out how many rotations we need
        # print('spacing: ', spacing, ' img shape: ', img.shape)
        # rotation_num = int(spacing[0] * img.shape[1] // 10)
        # slice_coverage = 10  # 10mm z-axis coverage
        # print('rotation_num: ', rotation_num, ' slice_coverage: ', slice_coverage)

        # # start to generate the motion for each rotation
        # # set whether static or motion for each rotation
 
        # while True:
        #     motion_status = [np.random.uniform(0,1) <motion_freq for k in range(rotation_num-2)]
        #     if motion_freq <=0:
        #         break
        #     if np.sum(motion_status) > 0:
        #         break
        # motion_status = [False] + motion_status + [False] # add two static rotations at the beginning and end
 
        # print('motion_status: ', motion_status)
        
        # amplitude_collect = np.zeros([rotation_num, CP_num, 6])
        # recon = np.zeros([slice_coverage * rotation_num,img.shape[2],img.shape[3]])
        # projection = np.zeros([slice_coverage * rotation_num, total_view, 1 , projector.nu])

        # for rot_n in range(0,rotation_num):
        #     # first set the SGA:
        #     sga = sga_reference + int(np.random.uniform(-5,5))
        #     sga = max(0, sga); sga = min(90, sga)
        #     sga_list.append(sga)
        #     # set the partial image:
        #     slice_start = rot_n * slice_coverage
        #     slice_end = (rot_n + 1) * slice_coverage
        #     slice_prior = max(0, slice_start - slice_coverage * 2)
        #     slice_post = min(img.shape[1], slice_end + slice_coverage * 2)
        #     print('slice start: ', slice_start, ' slice end: ', slice_end, ' slice prior: ', slice_prior, ' slice post: ', slice_post)
        #     img_partial = img[:,slice_prior:slice_post,:]

        #     # set the motion 
        #     # first: determine whether in this rotation we have motion or static, and whether it's severe or mild
        #     if motion_status[rot_n]:
        #         amplitude_max_tem_r = amplitude_max_mild 
        #         displacement_max_tem_r = displacement_max_mild

        #         if np.random.uniform(0,1) < severe_freq: # severe motion
        #             severe_motion = True
        #             amplitude_max_tem_t = amplitude_max_severe
        #             displacement_max_tem_t = displacement_max_severe
        #         else: # mild motion
        #             severe_motion = False
        #             amplitude_max_tem_t = amplitude_max_mild
        #             displacement_max_tem_t = displacement_max_mild
        #     else:
        #         severe_motion = False
        #         amplitude_max_tem_t, displacement_max_tem_t, amplitude_max_tem_r, displacement_max_tem_r = 0, 0, 0, 0
        #     print('severe motion: ', severe_motion)
        #     # second: determine whether the initial pose has been changed compared to last rotation
        #     offset_values = [0] * 6 # default
            
        #     # third: generate the motion
        #     # translation
        #     amplitude_txty_mm = transform.motion_control_point_generation(2, CP_num, amplitude_max = amplitude_max_tem_t, displacement_max = displacement_max_tem_t, change_direction_limit = change_direction_limit, offset_value = offset_values[0:2], print_result =False)
        #     amplitude_tx_mm = amplitude_txty_mm[:,0]
        #     amplitude_ty_mm = amplitude_txty_mm[:,1]
        #     if severe_motion:
        #         amplitude_tz_mm = transform.motion_control_point_generation(1, CP_num, amplitude_max = 3, displacement_max = 1.5, change_direction_limit = change_direction_limit, offset_value = offset_values[2], print_result =False)[:,0]
        #     elif not severe_motion and motion_status[rot_n]:
        #         amplitude_tz_mm = transform.motion_control_point_generation(1, CP_num, amplitude_max = 2, displacement_max = 1, change_direction_limit = change_direction_limit, offset_value = offset_values[2], print_result =False)[:,0]
        #     else:
        #         amplitude_tz_mm = transform.motion_control_point_generation(1, CP_num, amplitude_max = 0, displacement_max = 0, change_direction_limit = change_direction_limit, offset_value = offset_values[2], print_result =False)[:,0]
            
        #     # rotations
        #     while True:
        #         amplitude_rxryrz_degree = transform.motion_control_point_generation(3, CP_num, amplitude_max = amplitude_max_tem_r, displacement_max = displacement_max_tem_r, change_direction_limit = change_direction_limit, offset_value = offset_values[3:6], print_result =False)
        #         amplitude_rx_degree = amplitude_rxryrz_degree[:,0]
        #         amplitude_ry_degree = amplitude_rxryrz_degree[:,1]
        #         amplitude_rz_degree = amplitude_rxryrz_degree[:,2]
        #         if np.max(abs(amplitude_rx_degree - amplitude_rx_degree[0]))+ np.max(abs(amplitude_ry_degree - amplitude_ry_degree[0])) <= 6: # default
        #             break
       

        #     # let's also consider the double skull artifacts, espeically in the occipital bone
        #     if np.random.uniform(0,1) < double_skull_freq and motion_status[rot_n]:
        #         print('yes we have double skull')
        #         while True:
        #             amplitude_ty_mm = transform.motion_control_point_generation(1, CP_num, amplitude_max = 12, displacement_max = 8, change_direction_limit = change_direction_limit, offset_value = offset_values[1], print_result =False)[:,0]
        #             amplitude_tx_mm = transform.motion_control_point_generation(1, CP_num, amplitude_max = 5, displacement_max = 3, change_direction_limit = change_direction_limit, offset_value = offset_values[0], print_result =False)[:,0]
        #             amplitude_tz_mm = transform.motion_control_point_generation(1, CP_num, amplitude_max = 1, displacement_max = 0.5, change_direction_limit = change_direction_limit, offset_value = offset_values[2], print_result =False)[:,0]

        #             amplitude_rxryrz_degree = transform.motion_control_point_generation(3, CP_num, amplitude_max = 3, displacement_max = 1, change_direction_limit = change_direction_limit, offset_value = offset_values[3:6], print_result =False)
        #             amplitude_rx_degree = amplitude_rxryrz_degree[:,0]
        #             amplitude_ry_degree = amplitude_rxryrz_degree[:,1]
        #             amplitude_rz_degree = amplitude_rxryrz_degree[:,2]
        #             if np.max(abs(amplitude_ty_mm - amplitude_ty_mm[0])) >= 7: # default
        #                 break

        #     print('amplitude_tx_mm: ', amplitude_tx_mm)
        #     print('amplitude_ty_mm: ', amplitude_ty_mm)
        #     print('amplitude_tz_mm: ', amplitude_tz_mm)
        #     print('amplitude_rx_degree: ', amplitude_rx_degree)
        #     print('amplitude_ry_degree: ', amplitude_ry_degree)
        #     print('amplitude_rz_degree: ', amplitude_rz_degree)
            
        #     # save motion paramaters
        #     collect = np.stack([amplitude_tx_mm, amplitude_ty_mm, amplitude_tz_mm, amplitude_rx_degree, amplitude_ry_degree, amplitude_rz_degree], axis = 1)
        #     amplitude_collect[rot_n,...] = collect
            
        #     # prepare spline fit
        #     spline_tx = transform.interp_func(t, np.asarray([i/spacing[1] for i in amplitude_tx_mm]))
        #     spline_ty = transform.interp_func(t, np.asarray([i/spacing[2] for i in amplitude_ty_mm]))
        #     spline_tz = transform.interp_func(t, np.asarray([i/spacing[0] for i in amplitude_tz_mm]))
        #     spline_rx = transform.interp_func(t,np.asarray([i / 180 * np.pi for i in amplitude_rx_degree]))
        #     spline_ry = transform.interp_func(t,np.asarray([i / 180 * np.pi for i in amplitude_ry_degree]))
        #     spline_rz = transform.interp_func(t,np.asarray([i / 180 * np.pi for i in amplitude_rz_degree]))

        #     angles = ff.get_angles_zc(total_view, 360 ,sga)

        #     # if static and has the static image ref, then no need to do the projection
        #     if motion_status[rot_n] == False:
        #         if static_ref is not None:
        #             recon[slice_coverage * (rot_n) : slice_coverage * (rot_n + 1),...] = static_ref[slice_coverage*rot_n:slice_coverage*(rot_n+1),...]
        #             continue

        #     # generate forward projection
        #     projection_partial = ct.fp_w_spline_motion_model(img_partial, projector, angles, spline_tx, spline_ty, spline_tz, spline_rx, spline_ry, spline_rz, geometry, total_view = total_view, gantry_rotation_time = gantry_rotation_time, slice_num = None, increment = view_increment, order = 3)
        #     projection_partial = projection_partial[slice_start - slice_prior : slice_end - slice_prior,...]
            
        #     # generate backprojection
        #     recon_partial = ct.filtered_backporjection(projection_partial,angles,projector,fbp_projector, geometry, back_to_original_value=True)
        #     recon[slice_coverage * (rot_n) : slice_coverage * (rot_n + 1),...] = recon_partial


        # parameter_file = os.path.join(random_folder,'motion_parameters.npy')
        # np.save(parameter_file, np.array([[amplitude_collect],[sga_list], [t], [total_view], [gantry_rotation_time]], dtype=object))
           
        # # save recon
        # recon_nb_image = np.rollaxis(recon,0,3)  
        # nb.save(nb.Nifti1Image(recon_nb_image,img_affine), os.path.join(random_folder,'image_data','recon.nii.gz'))


        ####### resample recon to [1,1,2.5]mm
        img_1mm_file = nb.load(os.path.join(random_folder,'image_data','recon.nii.gz'))
        img_1mm = img_1mm_file.get_fdata()
        # original pixel dim
        pixel_dim = img_1mm_file.header.get_zooms()[:3]
        print('original pixel dim:', pixel_dim)
        
        # first, resample to z=1.25mm using interpolation, so that 2.5mm can be achieved by averaging
        new_dim = [1,1,1.25]
        hr_resample = dp.resample_nifti(img_1mm_file, order=3,  mode = 'nearest',  cval = np.min(img_1mm_file.get_fdata()), in_plane_resolution_mm=new_dim[0], slice_thickness_mm=new_dim[-1])
        hr_resample = nb.Nifti1Image(hr_resample.get_fdata(), affine=hr_resample.affine, header=hr_resample.header)

        # print('last shape:', hr_resample.get_fdata().shape)
        print('new pixel dim:', hr_resample.header.get_zooms()[:3])

        # now, do for z resampling (using averaging)
        img_data = hr_resample.get_fdata()
        affine = hr_resample.affine
        pixel_dim = hr_resample.header.get_zooms()[:3]
        header = hr_resample.header

        new_z_res = 2.5

        slice_factor = new_z_res // pixel_dim[-1]

        # Calculate the new shape
        new_shape = list(img_data.shape)
        new_shape[2] = int(img_data.shape[2] // slice_factor)
        print('new shape:', new_shape)  
        # Initialize the resampled data
        resampled_data = np.zeros(new_shape)

        for i in range(new_shape[2]):
            start_slice = int(i * slice_factor)
            end_slice = int((i + 1) * slice_factor)

            resampled_data[:, :, i] = np.mean(img_data[:, :, start_slice:end_slice], axis=2)

        # Update the affine and header for the new voxel size
        new_affine = affine.copy()
        new_affine[2, 2] = affine[2, 2] * slice_factor
        new_header = header.copy()
        new_header.set_zooms([1,1, pixel_dim[-1] * slice_factor])
        
        # Create and save the new NIfTI image
        resampled_img = nb.Nifti1Image(resampled_data, new_affine, new_header)
        # print new pixel dim:
        print('final pixel dim:', resampled_img.header.get_zooms()[:3])

        nb.save(resampled_img, os.path.join(random_folder,'image_data','recon_resample_avg.nii.gz'))


 0 00014689 0000455416
nib image shape:  (161, 239, 239)  spacing:  [1. 1. 1.]
static_ref is None?  True

 1 random
original pixel dim: (1.0, 1.0, 1.0)
new pixel dim: (1.0, 1.0, 1.25)
new shape: [239, 239, 64]
final pixel dim: (1.0, 1.0, 2.5)
